# 5주차 제출: RAGAS + LLM-as-Judge + A/B + Langfuse

1. Chroma DB 기반 RAGAS QA 쌍 42개 생성
2. GPT-4.1-mini / Gemini-3.1-flash-lite-preview 답변 생성
3. LLM-as-Judge 평가
4. A/B Pairwise 비교
5. Langfuse 연동 평가

In [1]:
from dotenv import load_dotenv
load_dotenv()

import os
import json
import ast
import pandas as pd
from typing import Any

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_chroma import Chroma

from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

from openevals.llm import create_llm_as_judge
from langfuse import get_client, Evaluation
from langfuse.langchain import CallbackHandler

required_env = [
    'OPENAI_API_KEY',
    'GOOGLE_API_KEY',
    'LANGFUSE_PUBLIC_KEY',
    'LANGFUSE_SECRET_KEY',
]
missing = [k for k in required_env if not os.getenv(k)]
if missing:
    raise ValueError(f'필수 환경변수 누락: {missing}')

print('환경변수 점검 완료')

c:\fun\002_etfbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


환경변수 점검 완료


In [2]:
# 공용 컴포넌트 초기화
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
langfuse_handler = CallbackHandler()

chroma_db = Chroma(
    collection_name='db_korean_cosine_metadata',
    embedding_function=embeddings,
    persist_directory='./chroma_db',
)

print(f'Chroma count: {chroma_db._collection.count()}')

Chroma count: 39


In [3]:
# Chroma 문서를 LangChain Document 리스트로 변환
raw = chroma_db.get()
docs: list[Document] = []

for i, txt in enumerate(raw.get('documents', [])):
    meta = raw.get('metadatas', [{}])[i] if i < len(raw.get('metadatas', [])) else {}
    docs.append(Document(page_content=txt or '', metadata=meta or {}))

if len(docs) == 0:
    raise ValueError('Chroma DB에서 문서를 찾지 못했습니다.')

print(f'문서 변환 완료: {len(docs)}개')
docs[0]

문서 변환 완료: 39개


Document(metadata={'company': '리비안(Rivian)', 'language': 'ko', 'source': 'data\\Rivian_KR.md'}, page_content='[출처] 이 문서는 리비안(Rivian)에 대한 문서입니다.\n----------------------------------\nRivian Automotive, Inc.는 2009년에 설립된 미국의 전기 자동차 제조업체, 자동차 기술 및 야외 레크리에이션 회사입니다.\n\n**주요 정보:**')

In [4]:
# 1) RAGAS로 QA 생성 (캐시 파일 있으면 재사용)
qa_cache_path = 'data/5week_ragas_qa_42.csv'
TESTSET_SIZE = 42
FORCE_REGENERATE = False

if os.path.exists(qa_cache_path) and not FORCE_REGENERATE:
    qa_df = pd.read_csv(qa_cache_path, encoding='utf-8-sig')
    print(f'기존 QA 캐시 로드: {qa_cache_path} ({len(qa_df)} rows)')
else:
    generator_llm = LangchainLLMWrapper(ChatOpenAI(model='gpt-4.1-mini', temperature=0))
    generator_embeddings = LangchainEmbeddingsWrapper(embeddings)

    testset_generator = TestsetGenerator(
        llm=generator_llm,
        embedding_model=generator_embeddings,
    )

    testset = testset_generator.generate_with_langchain_docs(
        docs[: min(len(docs), 250)],
        testset_size=TESTSET_SIZE,
    )

    qa_df = testset.to_pandas()
    qa_df.to_csv(qa_cache_path, index=False, encoding='utf-8-sig')
    print(f'신규 QA 생성 및 저장: {qa_cache_path} ({len(qa_df)} rows)')

qa_df.head(3)


C:\Users\kimdonggyu\AppData\Local\Temp\ipykernel_3776\4011536408.py:2: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model='gpt-4.1-mini', temperature=0))
C:\Users\kimdonggyu\AppData\Local\Temp\ipykernel_3776\4011536408.py:3: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(embeddings)
Generating Samples: 100%|██████████| 5/5 [00:09<00:00,  1.82s/it]


생성 완료: 5 rows


,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,리비안은 미국에서 어떤 종류의 회사인가요?,[[출처] 이 문서는 리비안(Rivian)에 대한 문서입니다.\n----------...,"리비안 Automotive, Inc.는 2009년에 설립된 미국의 전기 자동차 제조...",Financial and Strategic Analyst,WEB_SEARCH_LIKE,MEDIUM,single_hop_specific_query_synthesizer
1,"Could you provide a detailed overview of RIVN,...",[[출처] 이 문서는 리비안(Rivian)에 대한 문서입니다.\n----------...,"RIVN, or Rivian, is a publicly listed company ...",Electric Vehicle Market Analyst,PERFECT_GRAMMAR,LONG,single_hop_specific_query_synthesizer
2,What are the key features of Rivian's electric...,[<1-hop>\n\n[출처] 이 문서는 리비안(Rivian)에 대한 문서입니다.\...,Rivian produces electric pickup trucks based o...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer


In [12]:
# 유틸 함수
def safe_to_list(x: Any) -> list[str]:
    if x is None:
        return []
    if isinstance(x, list):
        return [str(v) for v in x]
    if isinstance(x, str):
        s = x.strip()
        if s.startswith('[') and s.endswith(']'):
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, list):
                    return [str(v) for v in parsed]
            except Exception:
                pass
        return [x]
    return [str(x)]

def get_col(df: pd.DataFrame, candidates: list[str]) -> str:
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f'컬럼을 찾지 못했습니다: {candidates}, 현재 컬럼={list(df.columns)}')

q_col = get_col(qa_df, ['user_input', 'question'])
ref_col = get_col(qa_df, ['reference', 'ground_truth', 'answer'])
ctx_col = get_col(qa_df, ['reference_contexts', 'contexts', 'retrieved_contexts'])

qa_df['question'] = qa_df[q_col].astype(str)
qa_df['reference'] = qa_df[ref_col].astype(str)
qa_df['reference_contexts'] = qa_df[ctx_col].apply(safe_to_list)

qa_df[['question', 'reference', 'reference_contexts']].head(2)

,question,reference,reference_contexts
0,리비안은 미국에서 어떤 종류의 회사인가요?,"리비안 Automotive, Inc.는 2009년에 설립된 미국의 전기 자동차 제조...",[[출처] 이 문서는 리비안(Rivian)에 대한 문서입니다.\n----------...
1,"Could you provide a detailed overview of RIVN,...","RIVN, or Rivian, is a publicly listed company ...",[[출처] 이 문서는 리비안(Rivian)에 대한 문서입니다.\n----------...


In [13]:
# 2) 두 모델 답변 생성 (GPT-4.1-mini vs Gemini-3.1-flash-lite-preview)
retriever = chroma_db.as_retriever(search_kwargs={'k': 4})

template = """당신은 RAG 기반 어시스턴트입니다.
주어진 Context를 우선 사용해 한국어로 정확하게 답변하세요.
Context에서 근거를 찾기 어렵다면 그렇게 명확히 말하세요.

[Context]
{context}

[Question]
{question}

[Answer]
"""

prompt = ChatPromptTemplate.from_template(template)
parser = StrOutputParser()

gpt_llm = ChatOpenAI(model='gpt-4.1-mini', temperature=0)
gemini_llm = ChatGoogleGenerativeAI(model='gemini-3.1-flash-lite-preview', temperature=0)

gpt_chain = prompt | gpt_llm | parser
gemini_chain = prompt | gemini_llm | parser

def format_docs(docs_: list[Document]) -> str:
    return '\n\n'.join(d.page_content for d in docs_)

rows = []
for _, r in qa_df.head(42).iterrows():
    q = r['question']
    ref = r['reference']

    retrieved = retriever.invoke(
        q,
        config={'callbacks': [langfuse_handler], 'tags': ['week5', 'retrieve']}
    )
    ctx_text = format_docs(retrieved)
    ctx_list = [d.page_content for d in retrieved]

    ans_a = gpt_chain.invoke(
        {'context': ctx_text, 'question': q},
        config={'callbacks': [langfuse_handler], 'tags': ['week5', 'answer', 'gpt']}
    )

    ans_b = gemini_chain.invoke(
        {'context': ctx_text, 'question': q},
        config={'callbacks': [langfuse_handler], 'tags': ['week5', 'answer', 'gemini']}
    )

    rows.append({
        'question': q,
        'reference': ref,
        'retrieved_contexts': ctx_list,
        'answer_gpt_4_1_mini': ans_a,
        'answer_gemini_2_5_flash': ans_b,
    })

ab_df = pd.DataFrame(rows)
ab_df.to_csv('data/5week_ab_raw_answers.csv', index=False, encoding='utf-8-sig')

print(ab_df.shape)
ab_df.head(2)

(5, 5)


,question,reference,retrieved_contexts,answer_gpt_4_1_mini,answer_gemini_2_5_flash
0,리비안은 미국에서 어떤 종류의 회사인가요?,"리비안 Automotive, Inc.는 2009년에 설립된 미국의 전기 자동차 제조...",[[출처] 이 문서는 리비안(Rivian)에 대한 문서입니다.\n----------...,리비안은 미국에 본사를 둔 상장 전기 자동차 제조업체입니다. NASDAQ에 RIVN...,"리비안(Rivian Automotive, Inc.)은 2009년에 설립된 미국의 전..."
1,"Could you provide a detailed overview of RIVN,...","RIVN, or Rivian, is a publicly listed company ...",[[출처] 이 문서는 리비안(Rivian)에 대한 문서입니다.\n----------...,리비안(Rivian)에 대한 상세 개요는 다음과 같습니다.\n\n1. **설립 및 ...,"제공된 정보를 바탕으로 리비안(Rivian, NASDAQ: RIVN)에 대해 정리한..."


In [14]:
# 3) LLM-as-Judge (단일 답변 절대평가)
SINGLE_JUDGE_PROMPT = """질문/컨텍스트/정답 기준으로 모델 답변을 0~1로 평가하세요.

기준:
- 정확성
- 근거 충실성(컨텍스트 기반)
- 완전성

<Question>
{input}
</Question>

<Context>
{context}
</Context>

<Reference>
{reference}
</Reference>

<Candidate Answer>
{prediction}
</Candidate Answer>

출력은 점수와 짧은 근거를 제공하세요.
"""

judge_single_gpt = create_llm_as_judge(
    prompt=SINGLE_JUDGE_PROMPT,
    feedback_key='single_score',
    continuous=True,
    model='openai:gpt-4.1-mini',
)

judge_single_gemini = create_llm_as_judge(
    prompt=SINGLE_JUDGE_PROMPT,
    feedback_key='single_score',
    continuous=True,
    model='google_genai:gemini-3.1-flash-lite-preview',
)

def run_single_eval(row: pd.Series, judge_fn, answer_col: str) -> dict:
    context_text = '\n\n'.join(safe_to_list(row['retrieved_contexts']))
    out = judge_fn(
        input=row['question'],
        prediction=row[answer_col],
        context=context_text,
        reference=row['reference'],
    )
    return {'score': float(out.get('score', 0.0)), 'comment': out.get('comment', '')}

for idx, row in ab_df.iterrows():
    gpt_judge_on_a = run_single_eval(row, judge_single_gpt, 'answer_gpt_4_1_mini')
    gpt_judge_on_b = run_single_eval(row, judge_single_gpt, 'answer_gemini_2_5_flash')

    gem_judge_on_a = run_single_eval(row, judge_single_gemini, 'answer_gpt_4_1_mini')
    gem_judge_on_b = run_single_eval(row, judge_single_gemini, 'answer_gemini_2_5_flash')

    ab_df.loc[idx, 'single_gpt_judge_score_a'] = gpt_judge_on_a['score']
    ab_df.loc[idx, 'single_gpt_judge_score_b'] = gpt_judge_on_b['score']
    ab_df.loc[idx, 'single_gemini_judge_score_a'] = gem_judge_on_a['score']
    ab_df.loc[idx, 'single_gemini_judge_score_b'] = gem_judge_on_b['score']

ab_df[['single_gpt_judge_score_a', 'single_gpt_judge_score_b', 'single_gemini_judge_score_a', 'single_gemini_judge_score_b']].describe()


,single_gpt_judge_score_a,single_gpt_judge_score_b,single_gemini_judge_score_a,single_gemini_judge_score_b
count,5.000000,5.000000,5.000000,5.0
mean,0.930000,0.970000,0.960000,1.0
std,0.027386,0.044721,0.054772,0.0
min,0.900000,0.900000,0.900000,1.0
25%,0.900000,0.950000,0.900000,1.0
50%,0.950000,1.000000,1.000000,1.0
75%,0.950000,1.000000,1.000000,1.0
max,0.950000,1.000000,1.000000,1.0


In [15]:
# 4) Pairwise A/B 테스트 (Judge 모델 2종)
PAIRWISE_PROMPT = """두 답변 A/B를 비교해 더 나은 쪽을 선택하세요.

평가 기준:
- 정확성
- 컨텍스트 근거성
- 완전성
- 간결성

<Question>
{input}
</Question>

<Context>
{context}
</Context>

<Reference>
{reference}
</Reference>

<Response A>
{prediction}
</Response A>

<Response B>
{prediction_b}
</Response B>

채점:
- 0.0: A 승
- 0.5: 무승부
- 1.0: B 승
"""

pairwise_gpt = create_llm_as_judge(
    prompt=PAIRWISE_PROMPT,
    feedback_key='pairwise',
    choices=[0.0, 0.5, 1.0],
    model='openai:gpt-4.1-mini',
)

pairwise_gemini = create_llm_as_judge(
    prompt=PAIRWISE_PROMPT,
    feedback_key='pairwise',
    choices=[0.0, 0.5, 1.0],
    model='google_genai:gemini-3.1-flash-lite-preview',
)

def score_to_label(s: float) -> str:
    if s == 0.0:
        return 'A_WIN_GPT_ANSWER'
    if s == 1.0:
        return 'B_WIN_GEMINI_ANSWER'
    return 'TIE'

for idx, row in ab_df.iterrows():
    context_text = '\n\n'.join(safe_to_list(row['retrieved_contexts']))

    r1 = pairwise_gpt(
        input=row['question'],
        prediction=row['answer_gpt_4_1_mini'],
        prediction_b=row['answer_gemini_2_5_flash'],
        context=context_text,
        reference=row['reference'],
    )
    r2 = pairwise_gemini(
        input=row['question'],
        prediction=row['answer_gpt_4_1_mini'],
        prediction_b=row['answer_gemini_2_5_flash'],
        context=context_text,
        reference=row['reference'],
    )

    s1 = float(r1.get('score', 0.5))
    s2 = float(r2.get('score', 0.5))

    ab_df.loc[idx, 'pairwise_judge_gpt_score'] = s1
    ab_df.loc[idx, 'pairwise_judge_gpt_label'] = score_to_label(s1)
    ab_df.loc[idx, 'pairwise_judge_gpt_comment'] = r1.get('comment', '')

    ab_df.loc[idx, 'pairwise_judge_gemini_score'] = s2
    ab_df.loc[idx, 'pairwise_judge_gemini_label'] = score_to_label(s2)
    ab_df.loc[idx, 'pairwise_judge_gemini_comment'] = r2.get('comment', '')

ab_df.to_csv('data/5week_ab_judged_42.csv', index=False, encoding='utf-8-sig')

summary = pd.DataFrame({
    'gpt_judge': ab_df['pairwise_judge_gpt_label'].value_counts(),
    'gemini_judge': ab_df['pairwise_judge_gemini_label'].value_counts(),
}).fillna(0).astype(int)

print('A/B 집계')
display(summary)


A/B 집계


,gpt_judge,gemini_judge
A_WIN_GPT_ANSWER,1,1
B_WIN_GEMINI_ANSWER,2,3
TIE,2,1


In [16]:
# 5) Langfuse 연동 평가 (run_experiment)
langfuse_client = get_client()

exp_data = []
for _, row in ab_df.iterrows():
    exp_data.append({
        'input': row['question'],
        'expected_output': row['reference'],
        'context': '\n\n'.join(safe_to_list(row['retrieved_contexts'])),
        'answer_a': row['answer_gpt_4_1_mini'],
        'answer_b': row['answer_gemini_2_5_flash'],
    })

def lf_task(*, item, **kwargs):
    return {
        'answer_a': item['answer_a'],
        'answer_b': item['answer_b'],
        'context': item['context'],
    }

def lf_pairwise_gpt(*, input, output, expected_output, **kwargs):
    try:
        res = pairwise_gpt(
            input=input,
            prediction=output['answer_a'],
            prediction_b=output['answer_b'],
            context=output['context'],
            reference=expected_output,
        )
        return Evaluation(
            name='pairwise_gpt_judge',
            value=float(res.get('score', 0.5)),
            comment=res.get('comment', '')[:300],
        )
    except Exception as e:
        return Evaluation(name='pairwise_gpt_judge', value=0.5, comment=str(e)[:300])

def lf_pairwise_gemini(*, input, output, expected_output, **kwargs):
    try:
        res = pairwise_gemini(
            input=input,
            prediction=output['answer_a'],
            prediction_b=output['answer_b'],
            context=output['context'],
            reference=expected_output,
        )
        return Evaluation(
            name='pairwise_gemini_judge',
            value=float(res.get('score', 0.5)),
            comment=res.get('comment', '')[:300],
        )
    except Exception as e:
        return Evaluation(name='pairwise_gemini_judge', value=0.5, comment=str(e)[:300])

langfuse_result = langfuse_client.run_experiment(
    name='W5_RAG_AB_42',
    description='42개 QA에 대해 GPT/Gemini 답변 A-B 비교 및 judge 2종 평가',
    data=exp_data,
    task=lf_task,
    evaluators=[lf_pairwise_gpt, lf_pairwise_gemini],
    max_concurrency=3,
)

print(langfuse_result)

In [17]:
# 최종 저장
ab_df.to_csv('data/5week_final_ab_eval.csv', index=False, encoding='utf-8-sig')

final_summary = {
    'num_rows': int(len(ab_df)),
    'single_gpt_judge_avg_a': float(ab_df['single_gpt_judge_score_a'].mean()),
    'single_gpt_judge_avg_b': float(ab_df['single_gpt_judge_score_b'].mean()),
    'single_gemini_judge_avg_a': float(ab_df['single_gemini_judge_score_a'].mean()),
    'single_gemini_judge_avg_b': float(ab_df['single_gemini_judge_score_b'].mean()),
    'pairwise_gpt_mean': float(ab_df['pairwise_judge_gpt_score'].mean()),
    'pairwise_gemini_mean': float(ab_df['pairwise_judge_gemini_score'].mean()),
}

with open('data/5week_summary.json', 'w', encoding='utf-8') as f:
    json.dump(final_summary, f, ensure_ascii=False, indent=2)

print('저장 완료:')
print('- data/5week_ragas_qa_42.csv')
print('- data/5week_ab_raw_answers.csv')
print('- data/5week_ab_judged_42.csv')
print('- data/5week_final_ab_eval.csv')
print('- data/5week_summary.json')
final_summary

저장 완료:
- data/5week_ragas_qa_42.csv
- data/5week_ab_raw_answers.csv
- data/5week_ab_judged_42.csv
- data/5week_final_ab_eval.csv
- data/5week_summary.json


{'num_rows': 5,
 'single_gpt_judge_avg_a': 0.9299999999999999,
 'single_gpt_judge_avg_b': 0.97,
 'single_gemini_judge_avg_a': 0.96,
 'single_gemini_judge_avg_b': 1.0,
 'pairwise_gpt_mean': 0.6,
 'pairwise_gemini_mean': 0.7}